# Evaluación cuantitativa: modelo base vs. Gemma-7b-it afinado con LoRA

Los ejemplos de `infer_gemma.ipynb` son útiles para "ver a ojo" si el
fine-tuning cambió algo, pero no tienen un resumen de referencia -así que no
se puede calcular ninguna métrica sobre ellos. Aquí usamos el split **`test`**
de `knkarthick/samsum` (resúmenes escritos por humanos, que el modelo **no**
vio durante el entrenamiento) para comparar el modelo base contra el
afinado con:

- **ROUGE-1 / ROUGE-2 / ROUGE-L / ROUGE-Lsum** — solapamiento de n-gramas
  contra la referencia (la métrica estándar para resumen automático).
- **BERTScore F1** — similitud semántica vía embeddings; a diferencia de
  ROUGE, reconoce paráfrasis ("compraron pan" vs. "fueron por pan" puntúan
  parecido aunque no comparten n-gramas).
- **Longitud promedio** del resumen generado, para ver si el modelo aprendió
  a ser tan conciso como la referencia humana (samsum pide 1-2 frases).

## Instalar dependencias adicionales


In [ ]:
%pip install evaluate rouge_score bert_score absl-py pandas


## 0. Configuración

In [ ]:
import os

MODEL_NAME = "google/gemma-7b-it"
ADAPTER_DIR = os.environ.get("OUTPUT_DIR", "/home/jovyan/labs/gemma-7b-it-samsum-lora")
DATASET_NAME = "knkarthick/samsum"
EVAL_SPLIT = "test"          # split de PRUEBA -no visto en el entrenamiento
N_EXAMPLES = 30               # empieza chico para iterar rápido; sube esto para un número más confiable
SEED = 42                     # para que la muestra sea reproducible
MAX_NEW_TOKENS = 64
BERTSCORE_MODEL_TYPE = "distilbert-base-uncased"  # chico (268MB); "roberta-large" (~1.4GB) es más preciso
OUTPUT_CSV = os.environ.get("EVAL_OUTPUT_CSV", "/home/jovyan/labs/eval_base_vs_finetuned.csv")

if not os.path.isdir(ADAPTER_DIR) or not os.listdir(ADAPTER_DIR):
    raise SystemExit(f"No encuentro adaptadores en {ADAPTER_DIR}. ¿Ya terminó el entrenamiento?")


## 1. Cargar el modelo base (4-bit) + los adaptadores LoRA

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("CUDA disponible:", torch.cuda.is_available())

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Modelo + adaptadores cargados.")


## 2. Funciones de generación (idénticas a `infer_gemma.ipynb`)

In [ ]:
def build_prompt(dialogue):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{dialogue}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def resumir(dialogue, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(dialogue)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def resumir_base(dialogue, max_new_tokens=MAX_NEW_TOKENS):
    with model.disable_adapter():
        return resumir(dialogue, max_new_tokens)


## 3. Tomar una muestra del split de prueba y generar con ambos modelos

Esto es lo que toma tiempo: dos generaciones (base + afinado) por cada
ejemplo. Con `N_EXAMPLES=30` en una L4 debería tomar unos minutos; si quieres
iterar más rápido, bájalo a 10-15 primero para revisar que todo corra bien.


In [ ]:
from datasets import load_dataset

test_dataset = load_dataset(DATASET_NAME, split=EVAL_SPLIT)
test_dataset = test_dataset.shuffle(seed=SEED).select(range(min(N_EXAMPLES, len(test_dataset))))
print(f"Evaluando sobre {len(test_dataset)} ejemplos de {DATASET_NAME} ({EVAL_SPLIT})")

dialogues, references = [], []
preds_finetuned, preds_base = [], []

for i, example in enumerate(test_dataset):
    dialogues.append(example["dialogue"])
    references.append(example["summary"])

    preds_finetuned.append(resumir(example["dialogue"]))
    preds_base.append(resumir_base(example["dialogue"]))

    if (i + 1) % 10 == 0 or (i + 1) == len(test_dataset):
        print(f"  ... {i + 1}/{len(test_dataset)} ejemplos generados")


## 4. Calcular ROUGE y BERTScore

`use_aggregator=False` en la segunda llamada da el ROUGE-L **por ejemplo**
(no solo el promedio) — lo guardamos en la sección 6 para poder encontrar
los mejores/peores casos de cada modelo.


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

rouge_finetuned = rouge.compute(predictions=preds_finetuned, references=references)
rouge_base = rouge.compute(predictions=preds_base, references=references)

rouge_finetuned_per_example = rouge.compute(
    predictions=preds_finetuned, references=references, use_aggregator=False
)
rouge_base_per_example = rouge.compute(
    predictions=preds_base, references=references, use_aggregator=False
)

print("ROUGE (base):     ", rouge_base)
print("ROUGE (afinado):  ", rouge_finetuned)


In [ ]:
bertscore = evaluate.load("bertscore")

bs_finetuned = bertscore.compute(
    predictions=preds_finetuned, references=references,
    lang="en", model_type=BERTSCORE_MODEL_TYPE,
)
bs_base = bertscore.compute(
    predictions=preds_base, references=references,
    lang="en", model_type=BERTSCORE_MODEL_TYPE,
)

bertscore_f1_base = sum(bs_base["f1"]) / len(bs_base["f1"])
bertscore_f1_finetuned = sum(bs_finetuned["f1"]) / len(bs_finetuned["f1"])

print("BERTScore F1 (base):    ", round(bertscore_f1_base, 4))
print("BERTScore F1 (afinado): ", round(bertscore_f1_finetuned, 4))


## 5. Tabla comparativa

In [ ]:
import pandas as pd

df_resumen = pd.DataFrame({
    "modelo": ["base (sin LoRA)", "afinado (con LoRA)"],
    "rouge1": [rouge_base["rouge1"], rouge_finetuned["rouge1"]],
    "rouge2": [rouge_base["rouge2"], rouge_finetuned["rouge2"]],
    "rougeL": [rouge_base["rougeL"], rouge_finetuned["rougeL"]],
    "rougeLsum": [rouge_base["rougeLsum"], rouge_finetuned["rougeLsum"]],
    "bertscore_f1": [bertscore_f1_base, bertscore_f1_finetuned],
    "longitud_promedio_palabras": [
        sum(len(p.split()) for p in preds_base) / len(preds_base),
        sum(len(p.split()) for p in preds_finetuned) / len(preds_finetuned),
    ],
})

referencia_len = sum(len(r.split()) for r in references) / len(references)
print(f"(longitud promedio de la referencia humana: {referencia_len:.1f} palabras)")
df_resumen


## 6. Guardar el detalle por ejemplo (para inspección manual)

Las columnas `rougeL_base` / `rougeL_afinado` te dejan ordenar y encontrar
los mejores y peores casos de cada modelo, en vez de solo mirar el promedio.


In [ ]:
df_detalle = pd.DataFrame({
    "dialogo": dialogues,
    "referencia": references,
    "resumen_base": preds_base,
    "resumen_afinado": preds_finetuned,
    "rougeL_base": rouge_base_per_example["rougeL"],
    "rougeL_afinado": rouge_finetuned_per_example["rougeL"],
})

df_detalle.to_csv(OUTPUT_CSV, index=False)
print(f"Detalle guardado en: {OUTPUT_CSV}")

# Los 3 ejemplos donde el modelo afinado mejoró más el ROUGE-L frente al base:
df_detalle["mejora"] = df_detalle["rougeL_afinado"] - df_detalle["rougeL_base"]
df_detalle.sort_values("mejora", ascending=False).head(3)[
    ["dialogo", "referencia", "resumen_base", "resumen_afinado", "mejora"]
]


## Notas finales

- **¿Por qué comparar contra el split `test` y no `train`?** Porque evaluar
  sobre ejemplos que el modelo ya vio durante el entrenamiento sobreestima
  qué tan bien generaliza -el punto de un split de prueba es medir
  desempeño en datos nuevos.
- **ROUGE vs. BERTScore:** si el modelo afinado mejora en ROUGE pero no en
  BERTScore (o viceversa), probablemente esté aprendiendo el *formato*
  (longitud, estilo telegráfico de samsum) más que el *contenido semántico*
  -vale la pena mirar `df_detalle` para casos concretos antes de concluir algo.
- **Extensión opcional (LLM-as-judge):** además de ROUGE/BERTScore, una
  práctica cada vez más común en 2026 es pedirle a un modelo más fuerte
  (Claude, GPT, Gemini) que califique cada resumen en una escala de
  fidelidad/coherencia frente al diálogo original -es más caro y más lento,
  pero puede detectar errores (alucinaciones, mezclar quién dijo qué) que
  ROUGE/BERTScore no ven porque solo miran similitud con la referencia.
- Este mismo pipeline existe como script plano en `evaluate_gemma.py`
  (con flags `--n_examples`, `--skip_bertscore`, etc.) para correrlo de una
  sola vez sin pasar por el notebook.
